# Sistema de Asistencia Inteligente - Unimarc
## Gestión de Inventario y Operaciones

In [4]:
import os
import time
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

#1. Cargar variables de entorno
load_dotenv()

#2. Configurar el modelo de lenguaje y la sesión de chat openai
llm = ChatOpenAI(
    base_url=os.environ["OPENAI_BASE_URL"],
    api_key=os.environ["GITHUB_TOKEN"],
    model="gpt-4o",
    temperature=0.7,
    streaming=True,
    max_tokens=4096,
    top_p=1
)

In [ ]:
#3. Crear una estructura para almacenar el historial de conversaciones por sesión
sesion_unimarc = {}

def historial_de_conversacion(sesion_id : str):
    if sesion_id not in sesion_unimarc:
        sesion_unimarc[sesion_id] = InMemoryChatMessageHistory()
    return sesion_unimarc[sesion_id]

#4. Función para sincronizar el contexto del historial de conversación, resumiendo si es necesario(VERSIÓN OPTIMIZADA)
def sincronizar_contexto_stock(sesion_id: str, max_mensajes=6):
    historial = historial_de_conversacion(sesion_id)

    if len(historial.messages) > max_mensajes:
        # Separamos los mensajes antiguos (a resumir) de los 2 más recientes
        mensajes_a_resumir = historial.messages[:-2]
        recientes = historial.messages[-2:]

        # Convertimos el historial antiguo a texto para que la IA lo lea
        conversation_text = ""
        for msj in mensajes_a_resumir:
            role = "Usuario" if msj.type == "human" else "Asistente"
            conversation_text += f"{role}: {msj.content}\n"
        
        # PROMPT DE ALTA EFICIENCIA: Obligamos a la IA a no perder datos importantes
        prompt_resumen = (
            "Eres un experto en logística de Unimarc. Resume la siguiente conversación. "
            "REGLA DE ORO: No omitas códigos de productos, nombres de artículos ni cantidades de stock. "
            "Resume el resto en máximo 2 líneas de forma técnica.\n\n"
            f"Historial a resumir:\n{conversation_text}"
        )

        # Invocamos el resumen con límite de tokens para ahorrar dinero/recursos
        summary_response = llm.invoke(prompt_resumen, max_tokens=150)
        summary = summary_response.content
        
        # Limpiamos el historial viejo y ponemos el nuevo resumen + los mensajes recientes
        historial.clear()
        historial.add_ai_message(f"[MEMORIA DE STOCK]: {summary}")
        historial.messages.extend(recientes)

#5. Crear el prompt de conversación con el contexto del historial
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente de Inventario de Unimarc. Ayudas a los usuarios a gestionar su inventario, responder preguntas sobre productos, y proporcionar información relevante sobre el stock y las operaciones de la tienda. Tienes que ser amigable, eficiente y siempre estar dispuesto a ayudar. Si no sabes la respuesta a una pregunta, es mejor admitirlo que inventar una respuesta incorrecta. Siempre debes mantener un tono profesional y cortés en tus respuestas. Las respuestas deben ser breves y al punto, evitando información innecesaria. Si el usuario hace una pregunta que no está relacionada con el inventario o las operaciones de la tienda, debes redirigir la conversación de vuelta a temas relevantes para Unimarc. Recuerda que tu objetivo principal es ayudar a los usuarios a gestionar su inventario de manera efectiva y proporcionar información precisa sobre los productos y operaciones de la tienda."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

conversation = RunnableWithMessageHistory(
    prompt | llm,
    historial_de_conversacion,
    input_messages_key="input",
    history_messages_key="chat_history"
)

#6. Función para ejecutar la conversación, sincronizando el contexto y mostrando la respuesta en tiempo real
def ejecutar_chat(input_text, session_id):
    sincronizar_contexto_stock(session_id)
    
    config = {"configurable": {"session_id": session_id}}
    
    print(f"[OUTPUT]: ", end="", flush=True)
    
    try:
        for chunk in conversation.stream({"input": input_text}, config=config):
            print(chunk.content, end="", flush=True)
        print("\n")
    except Exception as e:
        print(f"\n[ERROR_LOG]: {e}")

#7. Simulación de una sesión de chat en la terminal
id_actual = "SYS-LOG-01"
print(f"TERMINAL DE GESTIÓN UNIMARC | SESIÓN: {id_actual}")
print("-" * 50)

while True:
    user_input = input("[INPUT]: ")
    
    if user_input.lower() in ["exit", "quit", "salir"]:
        print("[SISTEMA]: Sesión finalizada.")
        break
    
    if user_input.strip():
        ejecutar_chat(user_input, id_actual)